# Débogage DINO — zone de 512 × 512 px, mode `tangent`

Tout se joue sur une zone de **`ZONE_PX × ZONE_PX` pixels** (512 par défaut, soit une cellule du niveau `LEVEL - 9`).
DINO y travaille sur de **petites images** : une par cellule de `PARENT_LEVEL`, de `TILE_PX` pixels de côté (64 par défaut).
Par défaut : cellules de 32 px (`PARENT_LEVEL = LEVEL - 5`) dans des images de 64 px, soit 4 × 4 tokens par image et 16 × 16 images dans la zone.
Une cellule HEALPix n'est pas un carré nord-sud : pour qu'elle tienne entière dans son image, il faut une image plus grande
que la cellule. C'est ce que mesure le test 1b.

| Test | Ce qu'on regarde |
|---|---|
| **0** | Niveau des données : valeurs brutes, et ce que DINO reçoit après `input_range` (image 8 bits ramenée à [0, 1]). |
| **1** | La zone en haut, et en dessous les images envoyées à DINO dans la même disposition, avec le contour de leur cellule. |
| **1b** | Contrôle chiffré : chaque pixel d'image comparé à la cellule HEALPix sous lui, et la part de chaque cellule qui tombe dans son image. |
| **1c** | Zoom sur une image : grille des patchs 16 px, nord en haut. |
| **1d** | `GetDINOV3SAT` sans stride : un vecteur DINO par image, UMAP, k-means → carte des classes à côté de l'image. Puis **kNN** : pour quelques images requêtes, leurs `KNN` plus proches voisins dans l'espace DINO et dans l'UMAP, en vignettes et sur la carte. |
| **3** | Oversampling des images, `GetDINOV3SAT(stride=1/OVERSAMPLE)` : on intercale des images centrées **entre** les centres de la grille, même projection tangente et même taille. Pour `OVERSAMPLE = 2`, on passe de 16 × 16 à 31 × 31 images dans la zone. Mêmes affichages que les tests 1 et 1d, kNN compris, avec les **mêmes** requêtes. |

Les tests 1d et 3 appellent directement `GetDINOV3SAT` : ils testent le package, pas une copie.


## Paramètres

In [ ]:
import os
SOURCE        = "g4e"        # "g4e" : produits GRID4EARTH  |  "synthetic" : motif connu, sans réseau
PRODUCT_INDEX = 0            # quel produit de G4E_PRODUCTS
LEVEL         = 20           # niveau HEALPix des données (~6 m)
ZONE_PX       = 512          # côté de la zone, en pixels de LEVEL (puissance de 2)
PARENT_LEVEL  = 15           # une image DINO par cellule de ce niveau, cellule de 2**(LEVEL-PARENT_LEVEL) px
                             #   15 -> cellules de 32 px, 16 x 16 images dans la zone
                             #   14 -> cellules de 64 px, 8 x 8 images
TILE_PX       = 64           # côté des images envoyées à DINO (multiple de 16)
                             #   64 avec des cellules de 32 px : la cellule tient entière dans son image
                             #   64 avec des cellules de 64 px : ~25 % de chaque cellule déborde (voir test 1b)
                             #   None = règle par défaut de GetDINOV3SAT (pow2_cover_px)
OVERSAMPLE    = 2            # test 3 : n*(GRID-1)+1 images par côté (16 -> 31 pour n = 2)
WEIGHTS       = os.path.expanduser("/home/jovyan/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth")
                             # poids DINOv3 SAT (.pth, comme dans dino_tuning_single_date) ou id HF
                             #   introuvable ou None -> backbone factice (couleur moyenne), annoncé en gros
INPUT_RANGE   = "auto"       # mise à l'échelle vers ce qu'attend DINO (image 8 bits / 255) :
                             #   "auto" (percentiles 1-99 de la zone), "reflectance" (0-0.3), "uint8", "unit", ou (lo, hi)
POOLING       = "cls"        # vecteur par image (GetDINOV3SAT) : "cls" | "mean" | "cls+mean"
N_CLUSTERS    = 8            # k-means sur l'UMAP
UMAP_NEIGHBORS = 15
KNN           = 8            # nombre de plus proches voisins affichés par image requête
N_QUERIES     = 4            # nombre d'images requêtes (une par classe, la plus typique)
QUERIES       = None         # ou une liste de (row, col) de la grille du test 1, ex. [(3, 4), (10, 12)]
EXCLUDE_OVERLAP = True       # kNN : ignorer les images qui chevauchent la requête (moins d'une cellule d'écart)
MODEL_NAME    = "dinov3_vitl16"
ZONE_LONLAT   = None         # (lon, lat) en degrés pour imposer la zone ; None = zone la plus complète
CACHE         = os.path.expanduser("/home/jovyan/s2_cache")   # réflectances sur disque (None pour désactiver)
STRETCH       = (2, 98)      # percentiles d'affichage, communs à TOUTES les vignettes
ZOOM_TILE     = None         # (row, col) de l'image à zoomer ; None = centre de la grille


In [ ]:
import sys, os, warnings
from contextlib import contextmanager
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

sys.path.insert(0, os.path.abspath(".."))      # healpix_analyse depuis le dépôt
sys.path.insert(0, os.path.abspath("."))       # g4e_source à côté du notebook
import healpix_analyse.dino as D

ZONE_LEVEL = LEVEL - int(np.log2(ZONE_PX))
k = LEVEL - PARENT_LEVEL                       # cellule parente = 2**k px
GRID = 2 ** (PARENT_LEVEL - ZONE_LEVEL)        # la zone fait GRID x GRID images
assert ZONE_PX & (ZONE_PX - 1) == 0 and ZONE_LEVEL <= PARENT_LEVEL
assert k >= D.DINOV3_PATCH_LEVELS, "LEVEL - PARENT_LEVEL >= 4 en mode tangent"
print(f"données niveau {LEVEL} (~{D.healpix_gsd_m(LEVEL):.1f} m) | zone niveau {ZONE_LEVEL} : "
      f"{ZONE_PX} px ~{D.healpix_gsd_m(ZONE_LEVEL)/1e3:.2f} km | cellules DINO niveau {PARENT_LEVEL} : "
      f"{2**k} px ~{D.healpix_gsd_m(PARENT_LEVEL):.0f} m, {GRID}x{GRID} dans la zone")

## Données

In [ ]:
def synthetic_source(level, zone_level, lon0=2.35, lat0=48.85):
    # Motif connu, défini dans un repère métrique EST / NORD, à l'échelle de la zone :
    #   routes N-S (blanches) et E-O (jaunes), disques rouges, une flèche bleue vers le NORD.
    # Sur une image correcte : routes N-S verticales, E-O horizontales, flèche vers le haut.
    zone = int(D.cells_at_lonlat(zone_level, np.array([lon0]), np.array([lat0]))[0])
    nb = D.cell_neighbours(zone_level, np.array([zone]))[:, 0]
    blocks = np.r_[zone, nb[nb >= 0]]
    kk = level - zone_level
    ids = ((blocks[:, None] << (2 * kk)) + np.arange(4 ** kk)[None]).ravel()
    lon, lat = D.cell_centres_lonlat(level, ids)
    zlon, zlat = D.cell_centres_lonlat(zone_level, np.array([zone]))
    xi, eta = D.lonlat_to_offsets(zlon, zlat, lon[None], lat[None])
    X, Y = xi[0] * D.EARTH_RADIUS_M, eta[0] * D.EARTH_RADIUS_M     # mètres, est / nord
    L = D.healpix_gsd_m(zone_level); g = D.healpix_gsd_m(level)
    rgb = np.empty((ids.size, 3), np.float32)
    rgb[:] = (0.08, 0.14, 0.06)
    rgb[:, 1] += 0.04 * np.sin(X / (0.09 * L)) * np.cos(Y / (0.12 * L))
    sx, sy = L / 6.5, L / 4.3
    rgb[np.abs((X + sx / 2) % sx - sx / 2) < 2.5 * g] = (0.35, 0.35, 0.35)
    rgb[np.abs((Y + sy / 2) % sy - sy / 2) < 1.8 * g] = (0.30, 0.28, 0.08)
    for cx, cy, r in [(0.20, 0.25, 0.04), (-0.30, 0.10, 0.06), (0.05, -0.30, 0.05), (-0.15, -0.20, 0.03)]:
        rgb[np.hypot(X - cx * L, Y - cy * L) < r * L] = (0.35, 0.06, 0.05)
    a = 0.12 * L
    shaft = (np.abs(X) < a * 0.25) & (Y > -a) & (Y < a)
    head = (Y >= a) & (Y < 2 * a) & (np.abs(X) < (2 * a - Y) * 0.8)
    rgb[shaft | head] = (0.05, 0.10, 0.40)
    rgb[np.hypot(X - 0.35 * L, Y + 0.35 * L) < 0.03 * L] = np.nan  # un trou : teste fill="mean"

    class S: pass
    s = S(); s.level, s.cell_id, s.ellipsoid, s.dates = level, ids, "sphere", ["synthétique"]
    s.rgb = lambda i, **kw: rgb
    s.zone_lonlat = (lon0, lat0)
    return s

if SOURCE == "g4e":
    from g4e_source import ProductSeries, G4E_PRODUCTS
    src = ProductSeries(G4E_PRODUCTS, level=LEVEL)
    rgb_all = src.rgb(PRODUCT_INDEX, cache=CACHE)
else:
    src = synthetic_source(LEVEL, ZONE_LEVEL)
    rgb_all = src.rgb(0)

D.set_ellipsoid(getattr(src, "ellipsoid", "sphere"))
ids_all = np.asarray(src.cell_id, dtype=np.int64)
print(f"{ids_all.size:,} cellules, ellipsoïde {D.HEALPIX_ELLIPSOID}")

## Choix de la zone et sous-ensemble

On garde la zone et ses 8 voisines du même niveau : les images DINO débordent de leur cellule.

In [ ]:
finite = np.isfinite(rgb_all).all(axis=1)
zk = 2 * (LEVEL - ZONE_LEVEL)
if ZONE_LONLAT is None and SOURCE == "synthetic":
    ZONE_LONLAT = src.zone_lonlat                          # la zone où est dessiné le motif
if ZONE_LONLAT is not None:
    zone = int(D.cells_at_lonlat(ZONE_LEVEL, np.array([ZONE_LONLAT[0]]), np.array([ZONE_LONLAT[1]]))[0])
else:
    zid, cnt = np.unique(ids_all[finite] >> zk, return_counts=True)
    full = cnt / 4 ** (LEVEL - ZONE_LEVEL)
    zl, za = D.cell_centres_lonlat(ZONE_LEVEL, zid)
    order = np.lexsort((np.hypot(zl - np.median(zl), za - np.median(za)), -np.round(full, 3)))
    zone = int(zid[order[0]])

znb = D.cell_neighbours(ZONE_LEVEL, np.array([zone]))[:, 0]
sel = np.isin(ids_all >> zk, np.r_[zone, znb[znb >= 0]])
ids, rgb = ids_all[sel], rgb_all[sel]
in_zone = (ids >> zk) == zone

g = PARENT_LEVEL - ZONE_LEVEL
kids = (zone << (2 * g)) + np.arange(GRID * GRID)                   # les cellules DINO de la zone
x, y = D.nested_to_xy(kids - (zone << (2 * g)))
ROW, COL = GRID - 1 - y, x                                          # convention de nested_to_tiles
zlon, zlat = D.cell_centres_lonlat(ZONE_LEVEL, np.array([zone]))
clon, clat = D.cell_centres_lonlat(PARENT_LEVEL, kids)
print(f"zone {zone} (niveau {ZONE_LEVEL}), centre {zlon[0]:.4f}E {zlat[0]:.4f}N, "
      f"couverture {np.isfinite(rgb[in_zone]).all(1).mean():.1%}")

lo, hi = np.nanpercentile(rgb[in_zone], STRETCH, axis=0)            # étirement commun
def show(a, chw=True):
    a = np.transpose(a, (1, 2, 0)) if chw else a
    return np.clip((np.nan_to_num(a, nan=float(lo.mean())) - lo) / (hi - lo), 0, 1)

## Test 0 — niveau des données en entrée de DINO

DINOv3 SAT a été entraîné sur des images RGB **8 bits divisées par 255**, puis normalisées par `SAT493M_MEAN / STD`. Le
réseau attend donc des valeurs dans [0, 1], avec la luminosité d'un produit visuel. Des données brutes (réflectance,
DN Sentinel-2, unités arbitraires...) lui donnent une image noire ou saturée, et des tokens qui ne portent presque rien.
`GetDINOV3SAT` ramène maintenant toujours l'entrée dans [0, 1] via `input_range`. Ici on fixe cette plage **une fois**
sur la zone, puis on la passe à tous les appels pour qu'ils voient exactement la même image.

In [ ]:
IN_LO, IN_HI, IN_HOW = D.input_range_of(rgb[in_zone], INPUT_RANGE)
scaled, _, _ = D.scale_input(rgb[in_zone], (IN_LO, IN_HI), verbose=True)
fig, axs = plt.subplots(1, 2, figsize=(13, 3.2))
for c_, name in enumerate("RGB"):
    v = rgb[in_zone][:, c_]; v = v[np.isfinite(v)]
    axs[0].hist(v, bins=200, histtype="step", color="rgb"[c_], label=name)
    w = scaled[:, c_]; w = w[np.isfinite(w)]
    axs[1].hist(255 * w, bins=256, range=(0, 255), histtype="step", color="rgb"[c_], label=name)
axs[0].axvline(IN_LO, c="k", ls="--"); axs[0].axvline(IN_HI, c="k", ls="--")
axs[0].set_title(f"valeurs brutes ; tirets = plage '{IN_HOW}' [{IN_LO:.4g}, {IN_HI:.4g}]")
axs[1].set_title("ce que DINO reçoit, en niveaux 0-255"); axs[1].legend()
plt.show()

def as_dino(img_chw):
    # image telle que le réseau la reçoit (avant la normalisation SAT), sans étirement d'affichage
    return np.clip((np.transpose(img_chw, (1, 2, 0)) - IN_LO) / (IN_HI - IN_LO), 0, 1)

## Les images DINO — mêmes primitives et mêmes paramètres que `GetDINOV3SAT(projection="tangent")`

In [ ]:
gsd = D.healpix_gsd_m(LEVEL)
px = gsd / D.EARTH_RADIUS_M
H = W = int(TILE_PX) if TILE_PX is not None else D.pow2_cover_px(kids, PARENT_LEVEL, gsd)
assert H % D.DINOV3_PATCH == 0

def tangent_tiles_for(centres, shift=(0.0, 0.0), fill="mean"):
    # D.tangent_tiles restreint aux cellules demandées
    cl, ca = D.cell_centres_lonlat(PARENT_LEVEL, np.asarray(centres))
    return tiles_at(cl, ca, shift=shift, fill=fill)

def tiles_at(cl, ca, shift=(0.0, 0.0), fill="mean"):
    # images tangentes nord-en-haut centrées en (cl, ca) : fonction du package
    return D.tangent_images(rgb, ids, LEVEL, cl, ca, (H, W), gsd_m=gsd, shift=shift, fill=fill)

tiles, valid, cov, tlon, tlat = tangent_tiles_for(kids)
print(f"{len(kids)} images de {H} x {W} px = {H // 16} x {W // 16} tokens, pour des cellules de {2**k} px")

# [1] contrôle : mêmes pixels que D.tangent_tiles (une image, données = son parent et son anneau)
c0 = kids[len(kids) // 2]
n0 = D.cell_neighbours(PARENT_LEVEL, np.array([c0]))[:, 0]
m0 = np.isin(ids >> (2 * k), np.r_[c0, n0[n0 >= 0]])
rt, rc, *_ = D.tangent_tiles(rgb[m0], ids[m0], LEVEL, PARENT_LEVEL, tile_px=(H, W), gsd_m=gsd, fill="nan")
mine, _, _, lon0, lat0 = tangent_tiles_for(np.array([c0]), fill="nan")
inside0 = (D.cells_at_lonlat(LEVEL, lon0[0], lat0[0]) >> (2 * k)) == c0
ok = np.isfinite(rt[rc == c0][0]) & np.isfinite(mine[0]) & inside0[None]
print("[1] identique à D.tangent_tiles :", np.allclose(rt[rc == c0][0][ok], mine[0][ok], atol=1e-6))

def outline_px(cell, level, c_lon, c_lat, Hh, Ww, pxx, step=8):
    # contour d'une cellule HEALPix en coordonnées pixel (col, row) d'une image tangente
    blon, blat = D._boundaries_lonlat(level, np.array([cell]), step=step)
    xi, eta = D.lonlat_to_offsets(np.atleast_1d(c_lon), np.atleast_1d(c_lat), blon, blat)
    return np.c_[xi[0] / pxx + (Ww - 1) / 2, (Hh - 1) / 2 - eta[0] / pxx]

# vue nord-en-haut de la zone à la résolution native : référence des tests 1 et 3
OH, OW = D.parent_cover_px(np.array([zone]), ZONE_LEVEL, gsd, margin_px=16)
olon, olat = D.tangent_grid_lonlat(zlon, zlat, (OH, OW), px)
olon, olat = olon[0], olat[0]
ov, ov_ok = D.sample_healpix(rgb, ids, LEVEL, olon, olat, interpolation="bilinear")
zone_mask = D.cells_at_lonlat(ZONE_LEVEL, olon, olat) == zone       # pixels de la vue dans la zone
print(f"vue nord-en-haut de la zone : {OH} x {OW} px")

## Test 1 — la zone, et en dessous les images vues par DINO

* **(a)** bloc NESTED exact de la zone (`ZONE_PX` px). C'est la disposition de la grille du dessous : la case `(r, c)` est la cellule de l'image `(r, c)`. Le nord est vers le **coin haut-droit**.
* **(b)** même zone, nord en haut, avec les cellules numérotées `r,c`.
* **Grille** : l'image `(r, c)` telle que DINO la reçoit (nord en haut, après `fill="mean"`). Contour rouge : sa cellule. Il doit contenir le même paysage que la case `(r, c)` de (a), et occuper la même place que dans (b).

In [ ]:
blk, _, _, _ = D.nested_to_tiles(rgb[in_zone], ids[in_zone], LEVEL, ZONE_LEVEL, fill="nan")
S = blk.shape[-1]

fig = plt.figure(figsize=(16, 25))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.9], hspace=0.06, wspace=0.05)
ax = fig.add_subplot(gs[0, 0]); ax.imshow(show(blk[0]))
ax.set_title(f"(a) bloc NESTED {S}x{S} px de la zone (nord ↗)")
s8 = S / GRID
for i in range(1, GRID):
    ax.axhline(i * s8 - .5, c="w", lw=.4, alpha=.6); ax.axvline(i * s8 - .5, c="w", lw=.4, alpha=.6)
for r, c in zip(ROW, COL):
    ax.text((c + .5) * s8, (r + .5) * s8, f"{r},{c}", c="w", fontsize=max(4, 56 // GRID), ha="center", va="center")
ax.set_xticks([]); ax.set_yticks([])

ax = fig.add_subplot(gs[0, 1]); ax.imshow(show(ov, chw=False))
ax.contour(zone_mask, levels=[.5], colors="w", linewidths=1)
ax.set_title("(b) même zone, nord en haut")
for cell, r, c, cl_, ca_ in zip(kids, ROW, COL, clon, clat):
    p = outline_px(cell, PARENT_LEVEL, zlon, zlat, OH, OW, px)
    ax.add_patch(Polygon(p, fill=False, ec="r", lw=.5, alpha=.7))
    ax.text(*p.mean(0), f"{r},{c}", c="w", fontsize=max(4, 56 // GRID), ha="center", va="center")
ax.set_xlim(-.5, OW - .5); ax.set_ylim(OH - .5, -.5); ax.set_xticks([]); ax.set_yticks([])

sub = gs[1, :].subgridspec(GRID, GRID, hspace=0.15, wspace=0.04)
for t, (r, c) in enumerate(zip(ROW, COL)):
    ax = fig.add_subplot(sub[r, c])
    ax.imshow(show(tiles[t]), interpolation="nearest")
    ax.add_patch(Polygon(outline_px(kids[t], PARENT_LEVEL, clon[t], clat[t], H, W, px), fill=False, ec="r", lw=.6))
    ax.set_xlim(-.5, W - .5); ax.set_ylim(H - .5, -.5)
    ax.set_title(f"{r},{c}", fontsize=max(4, 48 // GRID), pad=1); ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(f"zone {zone} — {GRID}x{GRID} images DINO de {H}x{W} px (cellules de {2**k} px, niveau {PARENT_LEVEL})", y=.9)
plt.show()

## Test 1b — contrôle chiffré

* **écart** : chaque pixel d'image comparé à la cellule HEALPix qui le contient. Faible et uniforme si tout va bien ; la même mesure sur l'image retournée haut/bas sert de référence.
* **part de la cellule dans l'image** : avec `TILE_PX = 2**k` (image de la même aire que la cellule), une cellule HEALPix n'est pas un carré et en déborde forcément. Ses tokens fins tombent alors hors de l'image et prennent le token du bord le plus proche.

In [ ]:
err, err_flip, frac_in = (np.full(len(kids), np.nan) for _ in range(3))
for t in range(len(kids)):
    near, nval = D.sample_healpix(rgb, ids, LEVEL, tlon[t], tlat[t], interpolation="nearest")
    tv, okp = np.transpose(tiles[t], (1, 2, 0)), valid[t] & nval
    if okp.any():
        err[t] = np.mean(np.abs(tv[okp] - near[okp]))
        err_flip[t] = np.mean(np.abs(tv[::-1][okp] - near[okp]))
    # part des pixels de la cellule (niveau LEVEL) qui tombent dans l'image
    ch = (kids[t] << (2 * k)) + np.arange(4 ** k)
    xi, eta = D.lonlat_to_offsets(clon[t:t+1], clat[t:t+1], *[v[None] for v in D.cell_centres_lonlat(LEVEL, ch)])
    cc, rr = xi[0] / px + (W - 1) / 2, (H - 1) / 2 - eta[0] / px
    frac_in[t] = np.mean((cc >= -.5) & (cc <= W - .5) & (rr >= -.5) & (rr <= H - .5))

fig, axs = plt.subplots(1, 2, figsize=(10, 4.5))
ge = np.full((GRID, GRID), np.nan); ge[ROW, COL] = err
im = axs[0].imshow(ge); plt.colorbar(im, ax=axs[0]); axs[0].set_title("écart moyen image / cellule")
gf = np.full((GRID, GRID), np.nan); gf[ROW, COL] = frac_in
im = axs[1].imshow(gf, vmin=0.5, vmax=1, cmap="RdYlGn"); plt.colorbar(im, ax=axs[1])
axs[1].set_title("part de la cellule dans son image")
plt.show()
print(f"écart moyen {np.nanmean(err):.4f} (image retournée : {np.nanmean(err_flip):.4f}) "
      f"| dynamique {np.nanmedian(hi - lo):.4f}")
print(f"part de la cellule dans son image : min {frac_in.min():.1%}, moyenne {frac_in.mean():.1%}")

## Test 1c — zoom sur une image

Grille jaune : les patchs 16 × 16 px, un token chacun. Rouge : la cellule. Pointillés magenta : pixels remplis par `fill="mean"`.

In [ ]:
zr, zc = ZOOM_TILE if ZOOM_TILE is not None else (GRID // 2, GRID // 2)
t = int(np.flatnonzero((ROW == zr) & (COL == zc))[0])
fig, axs = plt.subplots(1, 3, figsize=(19, 6.5))
axs[2].imshow(as_dino(tiles[t]), interpolation="nearest")
axs[2].set_title("la même image telle que DINO la reçoit\n(après input_range, sans étirement d'affichage)")
axs[2].set_xticks([]); axs[2].set_yticks([])
ax = axs[0]; ax.imshow(show(tiles[t]), interpolation="nearest")
for i in range(0, max(H, W) + 1, D.DINOV3_PATCH):
    ax.axhline(i - .5, c="y", lw=.6); ax.axvline(i - .5, c="y", lw=.6)
ax.add_patch(Polygon(outline_px(kids[t], PARENT_LEVEL, clon[t], clat[t], H, W, px), fill=False, ec="r", lw=1.5))
if (~valid[t]).any():
    ax.contour(~valid[t], levels=[.5], colors="m", linestyles=":", linewidths=.8)
ax.set_xlim(-.5, W - .5); ax.set_ylim(H - .5, -.5)
ax.set_title(f"image ({zr},{zc}) — ce que DINO reçoit ({H}x{W} px, nord en haut)")
mc = (ids >> (2 * k)) == kids[t]
b, _, _, _ = D.nested_to_tiles(rgb[mc], ids[mc], LEVEL, PARENT_LEVEL, fill="nan")
axs[1].imshow(show(b[0]), interpolation="nearest"); axs[1].set_title(f"la cellule en bloc NESTED {2**k} px (nord ↗)")
axs[1].set_xticks([]); axs[1].set_yticks([])
plt.show()

## Modèle DINO et outils communs aux tests 1d et 3

Les vecteurs sont calculés par `GetDINOV3SAT` lui-même (normalisation SAT-493M, `pooling=POOLING`), sur la zone et un
anneau autour. Les poids sont ceux de `WEIGHTS`, par défaut le fichier utilisé par `dino_tuning_single_date`. S'il est
introuvable, un backbone factice remplace DINOv3 : chaque token vaut
la couleur moyenne de son patch. Il sert à vérifier toute la chaîne (images → vecteurs → UMAP → classes → carte) sans
les poids. Ses classes ne regroupent alors que des couleurs.

In [ ]:
import torch, torch.nn as nn

class MeanDino(nn.Module):
    # backbone factice : token = couleur moyenne du patch 16x16, CLS = moyenne des tokens
    def __init__(self):
        super().__init__(); self.dummy = nn.Parameter(torch.zeros(1))
    def forward_features(self, x):
        tok = nn.functional.avg_pool2d(x, D.DINOV3_PATCH).flatten(2).transpose(1, 2)
        return {"x_norm_clstoken": tok.mean(1), "x_norm_patchtokens": tok}

_have = WEIGHTS is not None and (os.path.exists(str(WEIGHTS)) or not str(WEIGHTS).endswith(".pth"))
if _have:
    model, MODEL_LABEL = D.load_dinov3_sat(MODEL_NAME, weights=WEIGHTS), f"DINOv3 {MODEL_NAME}"
    print(f"{MODEL_LABEL} chargé depuis {WEIGHTS}")
else:
    model, MODEL_LABEL = MeanDino(), "backbone FACTICE (couleur moyenne) — PAS DINOv3"
    print("#" * 78)
    print(f"/!\\ poids introuvables ({WEIGHTS}) : backbone FACTICE.")
    print("    Les classes et les voisins ne reflètent que la couleur moyenne. Corriger WEIGHTS.")
    print("#" * 78)
model = model.eval()

# entrée de GetDINOV3SAT : les cellules de la zone et d'un anneau assez large pour que
# les images du bord de la zone voient leurs vraies données
ring = set(kids.tolist())
for _ in range(max(1, int(np.ceil(H / 2 ** k)))):
    nbr = D.cell_neighbours(PARENT_LEVEL, np.array(sorted(ring)))
    ring |= set(nbr[nbr >= 0].ravel().tolist())
m_in = np.isin(ids >> (2 * k), np.array(sorted(ring)))

def dino(stride=1.0):
    # appel direct du package : mêmes images, même normalisation, même pooling
    return D.GetDINOV3SAT(rgb[m_in], ids[m_in], LEVEL, PARENT_LEVEL, projection="tangent",
                          tile_px=TILE_PX, stride=stride, model=model, pooling=POOLING,
                          input_range=(IN_LO, IN_HI),            # la plage fixée au test 0, pour tous les appels
                          min_coverage=0.0, batch_size=64)

def rows_of(cell_id, wanted):
    # position de chaque id de `wanted` dans `cell_id`
    o = np.argsort(cell_id); p = np.searchsorted(cell_id[o], wanted)
    assert np.array_equal(cell_id[o][p], wanted), "images manquantes dans le résultat"
    return o[p]

def umap_kmeans(E, seed=0):
    import umap
    from sklearn.cluster import KMeans
    warnings.filterwarnings("ignore", message="n_jobs value")
    Z = umap.UMAP(n_neighbors=min(UMAP_NEIGHBORS, len(E) - 1), min_dist=0.1, metric="cosine",
                  random_state=seed).fit_transform(E)
    lab = KMeans(N_CLUSTERS, n_init=10, random_state=seed).fit_predict(Z)
    return Z, lab

CMAP = plt.get_cmap("tab10" if N_CLUSTERS <= 10 else "tab20", N_CLUSTERS)
def label_map(ax, lab_grid, extent, title):
    ax.imshow(lab_grid, cmap=CMAP, vmin=-.5, vmax=N_CLUSTERS - .5, extent=extent, interpolation="nearest")
    ax.set_xlim(0, S); ax.set_ylim(S, 0); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title)

def image_panel(ax, title="image de départ (bloc NESTED, nord ↗)"):
    ax.imshow(show(blk[0])); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title)

QCOLS = ["yellow", "cyan", "magenta", "orange", "lime", "white", "deepskyblue", "salmon"]

def knn_figure(E, Z, imgs, xy, queries, title):
    # Pour chaque requête : ses KNN voisins dans l'espace DINO (cosinus) et dans l'UMAP.
    #   E [M, D] vecteurs, Z [M, 2] UMAP, imgs [M, 3, H, W] images vues par DINO,
    #   xy [M, 2] position (x, y) des centres dans le bloc NESTED de la zone.
    En = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-12)
    nq = len(queries)
    fig = plt.figure(figsize=(22, 1.0 + 3.0 * nq))
    gk = fig.add_gridspec(1, 2, width_ratios=[1, 2.1], wspace=.03)
    axm = fig.add_subplot(gk[0, 0]); image_panel(axm, title)
    sub = gk[0, 1].subgridspec(2 * nq, KNN + 1, hspace=.45, wspace=.04)
    for qi, q in enumerate(queries):
        col = QCOLS[qi % len(QCOLS)]
        axm.scatter(*xy[q], s=260, marker="*", c=col, edgecolors="k", zorder=5)
        for row, (space, mk) in enumerate([("DINO", "o"), ("UMAP", "s")]):
            d = (1 - En @ En[q]) if space == "DINO" else np.linalg.norm(Z - Z[q], axis=1)
            d = d.astype(float).copy(); d[q] = np.inf
            if EXCLUDE_OVERLAP:
                d[np.hypot(*(xy - xy[q]).T) < 0.99 * s8] = np.inf
            nn_ = np.argsort(d)[:KNN]
            axm.scatter(*xy[nn_].T, s=60, marker=mk, facecolors="none", edgecolors=col, lw=1.6, zorder=4)
            a = fig.add_subplot(sub[2 * qi + row, 0]); a.imshow(show(imgs[q]), interpolation="nearest")
            for sp in a.spines.values(): sp.set_edgecolor(col); sp.set_linewidth(3)
            a.set_xticks([]); a.set_yticks([])
            a.set_ylabel(f"req. {qi + 1}\n{space}", fontsize=8)
            for j, t_ in enumerate(nn_):
                a = fig.add_subplot(sub[2 * qi + row, j + 1]); a.imshow(show(imgs[t_]), interpolation="nearest")
                a.set_title(f"cos {1 - d[t_]:.3f}" if space == "DINO" else f"d {d[t_]:.2f}", fontsize=7, pad=1)
                a.set_xticks([]); a.set_yticks([])
    fig.suptitle(f"{MODEL_LABEL} — {KNN} plus proches voisins (ligne DINO : cosinus sur les vecteurs bruts ; "
                 f"ligne UMAP : distance dans l'UMAP){' ; chevauchements exclus' if EXCLUDE_OVERLAP else ''}\n"
                 "carte : étoile = requête, rond = voisin DINO, carré = voisin UMAP", y=1.0, fontsize=10)
    plt.show()

## Test 1d — DINO sur chaque image, UMAP, k-means

Un vecteur DINO par image du test 1, soit `GRID × GRID` vecteurs. Ils sont projetés en 2-D avec UMAP (métrique cosinus), puis
regroupés en `N_CLUSTERS` classes par k-means sur les coordonnées UMAP, comme dans `dino_umap_sentinel2.py`.
À droite, chaque case `(r, c)` prend la couleur de la classe de son image. Les cases sont dans la disposition de l'image
de gauche.

Ensuite le **kNN** : pour `N_QUERIES` images requêtes (par défaut, l'image la plus typique de chacune des plus grandes
classes), leurs `KNN` plus proches voisins. Ligne « DINO » : similarité cosinus sur les vecteurs bruts, sans UMAP. Ligne
« UMAP » : distance dans l'UMAP. Si DINO fonctionne, les voisins se ressemblent par le **type de surface** (bâti, champ,
forêt, eau...), même loin de la requête et sous un éclairage un peu différent.

In [ ]:
r1 = dino(1.0)
E1 = r1.embedding[rows_of(r1.cell_id, kids)]                       # dans l'ordre des images du test 1
Z1, lab1 = umap_kmeans(E1)
print(f"{MODEL_LABEL} : {E1.shape[0]} images -> vecteurs de dimension {E1.shape[1]} (pooling '{POOLING}')")

L1 = np.full((GRID, GRID), -1); L1[ROW, COL] = lab1
fig, axs = plt.subplots(1, 2, figsize=(16, 8))
image_panel(axs[0])
for i in range(1, GRID):
    axs[0].axhline(i * s8 - .5, c="w", lw=.3, alpha=.5); axs[0].axvline(i * s8 - .5, c="w", lw=.3, alpha=.5)
label_map(axs[1], L1, (0, S, S, 0), f"{N_CLUSTERS} classes k-means sur l'UMAP — {GRID}x{GRID} images")
fig.suptitle(MODEL_LABEL, y=.95); plt.show()

# images requêtes : choisies par l'utilisateur, ou la plus typique (proche du centre UMAP) des plus grandes classes
if QUERIES is not None:
    Q1 = [int(np.flatnonzero((ROW == r) & (COL == c))[0]) for r, c in QUERIES]
else:
    big = np.argsort(-np.bincount(lab1, minlength=N_CLUSTERS))[:N_QUERIES]
    Q1 = [int(np.flatnonzero(lab1 == c_)[np.argmin(np.linalg.norm(Z1[lab1 == c_] - Z1[lab1 == c_].mean(0), axis=1))])
          for c_ in big]
Q_CELLS = kids[Q1]                                                   # mêmes requêtes au test 3
XY1 = np.c_[(COL + .5) * s8, (ROW + .5) * s8]
knn_figure(E1, Z1, tiles, XY1, Q1, f"sans stride : {GRID}x{GRID} images")


## Test 3 — oversampling des images : `GetDINOV3SAT(stride=1/OVERSAMPLE)`

Même projection (plan tangent nord-en-haut centré sur l'image), même résolution, même taille `TILE_PX`. Seule change la
grille des centres. Entre deux centres voisins de la grille `GRID × GRID` du test 1, on ajoute `OVERSAMPLE - 1` centres
régulièrement espacés : au milieu de deux voisins, au centre de quatre. Ces centres sont donnés par `D.stride_centres` :
ce sont des points exacts de la grille HEALPix, les coins nord des cellules du niveau `PARENT_LEVEL + log2(OVERSAMPLE)`. On obtient `OVERSAMPLE·(GRID-1)+1` images par
côté, 31 × 31 pour 16 × 16 et `OVERSAMPLE = 2`. Les images de rang pair sont exactement celles du test 1.

C'est une fenêtre glissante au niveau des **images** : le pas passe d'une cellule à une demi-cellule. Ce n'est pas `over_sample`,
qui décale de `16/n` px les fenêtres **à l'intérieur** de chaque image pour densifier les tokens.

Le package produit `OVERSAMPLE²` images par cellule, soit 32 × 32 pour la zone. La dernière ligne et la dernière colonne
sont les demi-pas vers la zone voisine ; on garde ici les 31 × 31 images intérieures.

**3a** — même figure que le test 1. Points rouges : centres du test 1 ; points cyan : centres ajoutés. Dans la mosaïque,
les images du test 1 ont un cadre rouge.

In [ ]:
n = int(OVERSAMPLE); mlev = int(np.log2(n))
GF = n * (GRID - 1) + 1
scid, slev, slon, slat, son = D.stride_centres(kids, PARENT_LEVEL, 1.0 / n)
par = rows_of(kids, scid >> (2 * mlev))                             # image -> rang de sa cellule
lx, ly = D.nested_to_xy(scid & (4 ** mlev - 1))
FJ = n * COL[par] + lx + 1 - n // 2                                 # position dans la grille fine
FI = n * ROW[par] - (ly + 1 - n // 2)
inner = (FI >= 0) & (FI < GF) & (FJ >= 0) & (FJ < GF)
scid, slon, slat, FI, FJ, orig, par = scid[inner], slon[inner], slat[inner], FI[inner], FJ[inner], son[inner], par[inner]
o = np.lexsort((FJ, FI))                                             # ordre ligne fine, colonne fine
scid, flon, flat, FI, FJ, orig, par = scid[o], slon[o], slat[o], FI[o], FJ[o], orig[o], par[o]

# contrôle de géométrie : un centre ajouté doit être à mi-chemin des centres voisins
Vc = np.zeros((GRID, GRID, 3)); Vc[ROW, COL] = D._lonlat_to_vec(clon, clat)
f = np.arange(GF) / n
i0 = np.minimum(np.floor(f).astype(int), GRID - 2); w = f - i0
wr, wc = w[:, None, None], w[None, :, None]
Vf = ((1 - wr) * (1 - wc) * Vc[i0][:, i0] + (1 - wr) * wc * Vc[i0][:, i0 + 1]
      + wr * (1 - wc) * Vc[i0 + 1][:, i0] + wr * wc * Vc[i0 + 1][:, i0 + 1])
Vf /= np.linalg.norm(Vf, axis=-1, keepdims=True)
dev_px = np.linalg.norm(D._lonlat_to_vec(flon, flat) - Vf[FI, FJ], axis=1) * D.EARTH_RADIUS_M / gsd
print(f"écart des centres stride_centres / milieux des centres voisins : max {dev_px.max():.2f} px")

tiles3, valid3, cov3, _, _ = tiles_at(flon, flat)
same = tiles3[np.flatnonzero(orig)[np.argsort(par[orig])]]          # images du test 1, ordre des cellules
print(f"{GF}x{GF} = {GF*GF} images de {H}x{W} px ; images du test 1 retrouvées à l'identique : "
      f"{np.allclose(same, tiles, atol=1e-5)} (écart max {np.abs(same - tiles).max():.2e})")
step_m = np.degrees(np.arccos(np.clip((Vf[0, 0] * Vf[0, 1]).sum() / np.linalg.norm(Vf[0, 0]) / np.linalg.norm(Vf[0, 1]), -1, 1)))
print(f"pas entre centres voisins ~{np.radians(step_m) * D.EARTH_RADIUS_M:.0f} m "
      f"(~{np.radians(step_m) * D.EARTH_RADIUS_M / gsd:.1f} px)")

fig = plt.figure(figsize=(16, 26))
gs3 = fig.add_gridspec(2, 2, height_ratios=[1, 1.9], hspace=0.06, wspace=0.05)
ax = fig.add_subplot(gs3[0, 0]); image_panel(ax, f"(a) bloc NESTED de la zone, {GF}x{GF} centres (nord ↗)")
ax.scatter((FJ / n + .5) * s8, (FI / n + .5) * s8, s=6, c=np.where(orig, "r", "c"))
ax = fig.add_subplot(gs3[0, 1]); ax.imshow(show(ov, chw=False))
ax.contour(zone_mask, levels=[.5], colors="w", linewidths=1)
xi, eta = D.lonlat_to_offsets(zlon, zlat, flon[None], flat[None])
ax.scatter(xi[0] / px + (OW - 1) / 2, (OH - 1) / 2 - eta[0] / px, s=6, c=np.where(orig, "r", "c"))
ax.set_xlim(-.5, OW - .5); ax.set_ylim(OH - .5, -.5); ax.set_xticks([]); ax.set_yticks([])
ax.set_title("(b) même zone, nord en haut")

# mosaïque : une image par centre, cadre rouge pour celles du test 1
g_ = 2; cell = H + 2 * g_
mos = np.full((GF * cell, GF * cell, 3), 0.15, np.float32)
for t_ in range(len(FI)):
    y0, x0 = FI[t_] * cell, FJ[t_] * cell
    mos[y0:y0 + cell, x0:x0 + cell] = (1, 0, 0) if orig[t_] else (0.85, 0.85, 0.85)
    mos[y0 + g_:y0 + g_ + H, x0 + g_:x0 + g_ + W] = show(tiles3[t_])
ax = fig.add_subplot(gs3[1, :]); ax.imshow(mos, interpolation="nearest"); ax.set_xticks([]); ax.set_yticks([])
ax.set_title(f"{GF}x{GF} images DINO de {H}x{W} px (cadre rouge : images du test 1)")
plt.show()

**3b** — DINO sur les `GF × GF` images, puis UMAP et k-means **sur l'ensemble**. Un seul ajustement, donc les mêmes classes
dans les deux cartes : au milieu, la grille du test 1 (les images de rang pair) ; à droite, la grille complète. Le
pas de la carte de droite est `OVERSAMPLE` fois plus fin.

In [ ]:
r3 = dino(1.0 / n)
E3 = r3.embedding[rows_of(r3.cell_id, scid)]                        # les GF x GF images, ordre de la grille
print(f"GetDINOV3SAT(stride=1/{n}) : {r3.embedding.shape[0]} images (zone + anneau), cell_level {r3.cell_level} ; "
      f"{len(E3)} dans la grille {GF}x{GF}")
e_on = E3[orig][np.argsort(par[orig])]
print("images du test 1 : mêmes vecteurs qu'avec stride=1 :", np.allclose(e_on, E1, atol=1e-4))
Z3, lab3 = umap_kmeans(E3)
L3 = np.full((GF, GF), -1); L3[FI, FJ] = lab3
half = 0.5 / n
fig, axs = plt.subplots(1, 3, figsize=(24, 8))
image_panel(axs[0])
label_map(axs[1], L3[::n, ::n], (0, S, S, 0), f"grille du test 1 : {GRID}x{GRID}")
label_map(axs[2], L3, ((0.5 - half) * s8, (GRID - 0.5 + half) * s8, (GRID - 0.5 + half) * s8, (0.5 - half) * s8),
          f"oversampling x{n} : {GF}x{GF}")
fig.suptitle(f"{MODEL_LABEL} — {N_CLUSTERS} classes k-means sur l'UMAP des {GF*GF} images", y=.98)
plt.show()

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(*Z3[~orig].T, c=lab3[~orig], cmap=CMAP, vmin=-.5, vmax=N_CLUSTERS - .5, s=8, label="centres ajoutés")
ax.scatter(*Z3[orig].T, c=lab3[orig], cmap=CMAP, vmin=-.5, vmax=N_CLUSTERS - .5, s=18, edgecolors="k", lw=.5,
           label="centres du test 1")
ax.legend(fontsize=8); ax.set_title("UMAP des images oversamplées"); plt.show()

**3c** — kNN avec `stride=1/2`, sur les mêmes requêtes que le test 1d. Il y a maintenant `GF²` candidats au lieu de `GRID²`,
et les images intermédiaires peuvent apporter des voisins mieux centrés sur le même objet. Avec `EXCLUDE_OVERLAP = False`,
les premiers voisins sont surtout les images décalées d'une demi-cellule, ce qui mesure la stabilité de DINO à un petit
décalage.

In [ ]:
Q3 = [int(np.flatnonzero(orig & (scid >> (2 * mlev) == c_))[0]) for c_ in Q_CELLS]
XY3 = np.c_[(FJ / n + .5) * s8, (FI / n + .5) * s8]
knn_figure(E3, Z3, tiles3, XY3, Q3, f"stride=1/{n} : {GF}x{GF} images")

# stabilité au décalage : similarité cosinus entre chaque image du test 1 et ses voisines à un demi-pas
En3 = E3 / np.linalg.norm(E3, axis=1, keepdims=True)
pos = {(i, j): t_ for t_, (i, j) in enumerate(zip(FI, FJ))}
sims = [float(En3[t_] @ En3[pos[(i + di, j + dj)]]) for t_, (i, j) in enumerate(zip(FI, FJ)) if orig[t_]
        for di, dj in [(0, 1), (1, 0), (1, 1)] if (i + di, j + dj) in pos]
rnd = En3[np.random.default_rng(0).integers(0, len(En3), 2000)]
print(f"cosinus moyen image / voisine à un demi-pas : {np.mean(sims):.3f}   "
      f"(paires d'images au hasard : {np.mean(np.sum(rnd[:1000] * rnd[1000:], 1)):.3f})")